# 08. Comprobar el modelo servido

Este notebook consulta el estado del modelo `invoice-review` y solicita exactamente una predicción con ocho features sintéticas. No entrena, no carga artifacts, no llama a MLflow ni inicia servicios. Tampoco cambia aliases ni ejecuta operaciones administrativas.

## Preparación

Use el entorno instalado y el kernel **InvoiceOps ML Python 3.12** descritos en el README. Antes de iniciar Jupyter, defina `MODEL_API_URL` con la URL base aprobada de una Model API ya disponible. No hay valor predeterminado ni lectura automática de `.env`. No escriba la URL ni credenciales en este notebook. Se admite HTTP local configurado explícitamente y HTTPS, con un prefijo de ruta opcional.

Ejecute las celdas en orden. El cliente no envía autenticación, cookies ni credenciales de MLflow; un despliegue que exija autenticación devolverá un error HTTP. No se siguen redirecciones, proxies del entorno ni reintentos. Cada respuesta tiene un límite inclusivo de 64 KiB y cada operación de socket un timeout de 10 segundos, no un plazo total.

In [ ]:
from invoiceops_ml.serving import ServingSmokeError, run_serving_smoke

## Estado y predicción

Primero se valida `GET /models/invoice-review/health`; sólo si tiene éxito se envía `POST /models/invoice-review/predict`. Cada respuesta conserva su propio `model_id`, `model_version` y `run_id`. Si el servidor cambia de modelo entre consultas, se muestran ambos snapshots y una advertencia: las llamadas no son atómicas. La probabilidad debe ser numérica, finita y estar entre 0 y 1; no constituye por sí sola una decisión de negocio.

In [ ]:
try:
    result = run_serving_smoke()
except ServingSmokeError as error:
    print(f"Comprobación fallida. Categoría: {error.category}")
else:
    for key, label in (("health", "Estado del modelo"), ("prediction", "Predicción")):
        snapshot = result[key]
        print(label)
        for field in ("model_id", "model_version", "run_id"):
            print(f"  {field}: {snapshot[field]}")
    print(f"Probabilidad: {result['prediction']['probability']}")
    if result["warnings"]:
        print("Advertencia: el modelo cambió entre consultas; los snapshots no son atómicos.")

## Interpretar y verificar

Las categorías seguras son `configuration`, `transport`, `http`, `schema` y `response_too_large`. No se imprimen URL, cabeceras, cuerpos de error ni excepciones del transporte. Ante un fallo, revise la configuración o disponibilidad por el canal autorizado, sin agregar credenciales al notebook.

La evidencia automatizada usa transporte simulado y ejecución de celdas en proceso: no verifica un servicio real ni un kernel independiente. Para QA manual autorizado, confirme los metadatos de ambas respuestas y la probabilidad; revise también una configuración ausente y un servicio no disponible. Borre todas las salidas antes de guardar o compartir. Para revertir, deje de ejecutar este notebook y retire únicamente el cliente y la sección del README de ML-15; no hay estado remoto que restaurar.